In [ ]:
# Run once per (training set, model). Settings used for the released corpora in icl_corpus/:
#   Qwen/Qwen3-1.7B           WARMUP_GENERATIONS = 2  (thinking mode; only the text after </think> is kept)
#   Qwen/Qwen2.5-7B-Instruct  WARMUP_GENERATIONS = 5
# Training sets: data/train_DAPO-Math-17k.parquet and data/train_deepcoder.parquet (full sets, see nbs/*_data.ipynb).
TRAIN_DATASET = "data/train_DAPO-Math-17k.parquet"
MODEL = "Qwen/Qwen3-1.7B"
WARMUP_GENERATIONS = 2
SAVE_FILE = "icl_corpus/train_DAPO-Math-17k_icl_corpus_qwen3_1.7b.json"

# Raw warm-up generations are cached here so that scoring can be re-run without regenerating.
GEN_CACHE = SAVE_FILE.replace(".json", "_generations.npy")

In [ ]:
import os
import json
import datasets
import numpy as np
from tqdm.auto import tqdm
from transformers import AutoTokenizer
from verl.utils.reward_score import default_compute_score
from concurrent.futures import ProcessPoolExecutor, as_completed

def filter_tests(tests):
    indexed = [(i, t) for i, t in enumerate(tests)]
    indexed.sort(key=lambda it: (-len(str(it[1].get("input"))), it[0]))
    selected_tests = [t for _, t in indexed[:15]]
    return selected_tests

def reward_model_processor(reward_models):
    processed_reward_models = []
    for reward_model in reward_models:
        if type(reward_model) == str:
            reward_model = json.loads(reward_model)
            reward_model["ground_truth"] = filter_tests(reward_model["ground_truth"])
            reward_model["ground_truth"] = json.dumps(reward_model["ground_truth"])
        processed_reward_models.append(reward_model)
    return {"reward_model": processed_reward_models}

dataframe = datasets.load_dataset("parquet", data_files=TRAIN_DATASET)["train"]
dataframe = dataframe.map(reward_model_processor,
    input_columns="reward_model",
    batched=True, batch_size=16,
    num_proc=8,
    desc="Processing reward_model"
)
df = dataframe.to_pandas()


In [ ]:
from vllm import LLM, SamplingParams

if os.path.exists(GEN_CACHE):
    gen_outputs = np.load(GEN_CACHE, allow_pickle=True)
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL)
    prompts = df["prompt"].apply(lambda x: tokenizer.apply_chat_template(list(x), tokenize=False, add_generation_prompt=True, enable_thinking=True)).tolist()

    llm = LLM(model=MODEL, tensor_parallel_size=4)
    sampling_params = SamplingParams(
        temperature=1.0, top_p=0.95, n=WARMUP_GENERATIONS, max_tokens=16384
    )
    outputs = llm.generate(prompts, sampling_params)
    gen_outputs = np.array(
        [[output.outputs[idx].text for idx in range(WARMUP_GENERATIONS)] for output in outputs], dtype=object
    )
    np.save(GEN_CACHE, gen_outputs)

In [ ]:
## Remove <think> ... </think>
if MODEL == "Qwen/Qwen3-1.7B":
    for idx in range(len(gen_outputs)):
        for gen_idx in range(WARMUP_GENERATIONS):
            if "</think>" in gen_outputs[idx][gen_idx]:
                gen_outputs[idx][gen_idx] = str(gen_outputs[idx][gen_idx].split("</think>")[-1].strip())
            else:
                gen_outputs[idx][gen_idx] = ""

In [ ]:
user_prompts = df["prompt"].apply(lambda x: x[0]["content"].strip()).values
ground_truths = df["reward_model"].apply(lambda x: x["ground_truth"]).values
data_source = df["data_source"].values

In [ ]:
def _score_one_prompt(args):
    idx, user_prompt, data_src, gt_raw, outs = args
    user_prompt = user_prompt.strip()
    passed = []
    for o in outs:
        try:
            if default_compute_score(data_src, o, gt_raw)==1:
                passed.append(str(o))
        except Exception:
            continue
    return user_prompt, passed

def build_icl_corpus_mp(user_prompts, ground_truths, gen_outputs, data_source,
                        max_workers=None):
    assert len(user_prompts) == len(ground_truths) == len(gen_outputs) == len(data_source)
    n = len(user_prompts)
    items = [(i, user_prompts[i], data_source[i], ground_truths[i], gen_outputs[i]) for i in range(n)]

    if max_workers is None:
        max_workers = os.cpu_count()-2 or 1

    icl_corpus = {}
    with ProcessPoolExecutor(max_workers=max_workers) as ex:
        futures = [ex.submit(_score_one_prompt, it) for it in items]
        with tqdm(total=n, desc="Scoring", dynamic_ncols=True) as pbar:
            for fut in as_completed(futures):
                try:
                    prompt, passed = fut.result()
                    icl_corpus[prompt] = passed
                except Exception as e:
                    print(e)
                    # keep going even if a task failed
                    pass
                pbar.update(1)
    return icl_corpus

In [ ]:
icl_corpus = build_icl_corpus_mp(user_prompts, ground_truths, gen_outputs, data_source)

In [ ]:
sum([len(v)>0 for k, v in icl_corpus.items()])/len(icl_corpus)

In [ ]:
import json
with open(SAVE_FILE, 'w') as f:
    json.dump(icl_corpus, f)